# Segmentation-Guided ResNet-50 — Google Colab

This notebook runs `003_classification/segmentation_guided_cv_resnet50/train.py` on a Colab GPU. The dataset is read from a single `dataset_gcolab.tar.gz` archive in Google Drive, while only the required content is extracted to Colab local storage: CV metadata, `ct_windowed`, ground-truth masks, and the selected U-Net probability maps. Training outputs are written directly to Google Drive so they persist after the runtime ends.

Before opening Colab, create the archive from the repository root:

```bash
tar -czf dataset_gcolab.tar.gz \
  000_dataset/_segmentation_dataset_v2/004_classification_cv_5fold_seed42.csv \
  000_dataset/_segmentation_dataset_v2/ct_windowed \
  000_dataset/_segmentation_dataset_v2/mask \
  experiment_results/dc730a13-5813-4d87-b15c-3b630deb32b5/segmentation/unet/inference/probability_npy
sha256sum dataset_gcolab.tar.gz > dataset_gcolab.tar.gz.sha256
```

Upload both files to the Google Drive folder configured in section 3. Make sure the source-code changes have also been pushed to the selected GitHub branch.

> Training runs five folds and may take longer than a single Colab runtime. The script saves a checkpoint after every epoch, but it does not yet provide a CLI for resuming an interrupted CV run. Use a sufficiently long runtime and keep the session open during training.

## 1. Enable and verify the GPU

Select **Runtime > Change runtime type > GPU** before running this cell.

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Enable it through Runtime > Change runtime type."
    )

disk = shutil.disk_usage("/content")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print(f"Disk free: {disk.free / 2**30:.1f} GiB")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Runtime configuration

Adjust the archive location in Google Drive. `EXPERIMENT_ID` must identify the experiment containing `segmentation/unet/inference/probability_npy` inside the archive. Reduce `BATCH_SIZE` if CUDA runs out of memory.

In [ ]:
from pathlib import Path

# Source code
REPOSITORY_URL = "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Archive uploaded to Google Drive
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/mask-guided-lung-nodule-xai")
DRIVE_DATASET_ARCHIVE = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz"
DRIVE_DATASET_CHECKSUM = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz.sha256"

# Extraction location on Colab local storage
LOCAL_DATA_ROOT = Path("/content/classification_data")
EXPERIMENT_ID = "dc730a13-5813-4d87-b15c-3b630deb32b5"

# Persistent checkpoints and metrics
DRIVE_EXPERIMENT_ROOT = (
    DRIVE_PROJECT_DIR / "experiment_results" / EXPERIMENT_ID
)
DRIVE_GUIDED_OUTPUT_DIR = (
    DRIVE_EXPERIMENT_ROOT / "classification/guided_resnet50"
)

# Colab training overrides
BATCH_SIZE = 32       # reduce to 16, 8, or 4 if an OOM occurs
NUM_WORKERS = 0  # stable on Colab/Python 3.13
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20

# Set to True only when re-extracting data in the same runtime
FORCE_REEXTRACT = False

print(f"Archive    : {DRIVE_DATASET_ARCHIVE}")
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Output     : {DRIVE_GUIDED_OUTPUT_DIR}")

## 4. Clone or update the repository

The GitHub branch must already contain `003_classification/segmentation_guided_cv_resnet50`.

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

required_script = (
    PROJECT_ROOT
    / "003_classification/segmentation_guided_cv_resnet50/train.py"
)
if not required_script.is_file():
    raise FileNotFoundError(
        f"Training script not found: {required_script}. "
        "Push the local changes to the GitHub branch first."
    )
print(f"Repository ready: {PROJECT_ROOT}")

## 5. Install dependencies

The Torch and Torchvision versions bundled with Colab are retained for CUDA compatibility. Zennit is required when running LRP from `test.py`.

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import sys

required_versions = {
    "albumentations": "2.0.8",
    "zennit": "0.5.1",
}
packages = []
for package, required_version in required_versions.items():
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        packages.append(f"{package}=={required_version}")

if packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )

required_modules = (
    "torch", "torchvision", "albumentations", "cv2", "numpy",
    "pandas", "sklearn", "matplotlib", "tqdm",
)
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Required dependencies are unavailable: {missing}")
print("All dependencies are ready.")

## 6. Validate and extract `dataset_gcolab.tar.gz`

The complete archive contains data that classification does not use. This cell extracts only:

- `004_classification_cv_5fold_seed42.csv`
- `ct_windowed/`
- ground-truth nodule masks from `mask/`
- probability maps from `experiment_results/EXPERIMENT_ID/segmentation/unet`

The archive is read directly from Drive to avoid creating another archive copy on local storage.

In [ ]:
import hashlib

if not DRIVE_DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(f"Archive not found: {DRIVE_DATASET_ARCHIVE}")

print(f"Archive size: {DRIVE_DATASET_ARCHIVE.stat().st_size / 2**30:.2f} GiB")

# The checksum is optional but recommended for large uploads.
if DRIVE_DATASET_CHECKSUM.is_file():
    expected_hash = DRIVE_DATASET_CHECKSUM.read_text().split()[0].strip().lower()
    digest = hashlib.sha256()
    with DRIVE_DATASET_ARCHIVE.open("rb") as archive_file:
        for chunk in iter(lambda: archive_file.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    actual_hash = digest.hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError("Archive checksum mismatch; the upload may be corrupted.")
    print("SHA-256 checksum is valid.")
else:
    print("Checksum not found; SHA-256 validation skipped.")

metadata_member = (
    "000_dataset/_segmentation_dataset_v2/"
    "004_classification_cv_5fold_seed42.csv"
)
ct_member = (
    "000_dataset/_segmentation_dataset_v2/ct_windowed"
)
mask_member = (
    "000_dataset/_segmentation_dataset_v2/mask"
)
probability_member = (
    f"experiment_results/{EXPERIMENT_ID}/"
    "segmentation/unet/inference/probability_npy"
)
required_members = (
    metadata_member, ct_member, mask_member, probability_member
)
extract_marker = (
    LOCAL_DATA_ROOT / f".classification_xai_extract_{EXPERIMENT_ID}"
)

if FORCE_REEXTRACT or not extract_marker.is_file():
    print("Extracting the required training data...", flush=True)
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "tar", "--checkpoint=5000",
            "--checkpoint-action=echo=Extract checkpoint %u",
            "-xzf", str(DRIVE_DATASET_ARCHIVE),
            "-C", str(LOCAL_DATA_ROOT), *required_members,
        ],
        check=True,
    )
    extract_marker.touch()
    print("Extraction complete.")
else:
    print("Local data has already been extracted; reusing it.")

## 7. Link local data and Google Drive outputs

Symlinks provide the dataset layout expected by `train.py` without duplicating data. Guided-classification outputs are written directly to `experiment_results/<UUID>/classification/guided_resnet50` in Google Drive.

In [ ]:
import os

local_dataset_dir = LOCAL_DATA_ROOT / "000_dataset"
local_experiment_dir = (
    LOCAL_DATA_ROOT / "experiment_results" / EXPERIMENT_ID
)
local_segmentation_component = local_experiment_dir / "segmentation"

for required_dir in (local_dataset_dir, local_segmentation_component):
    if not required_dir.is_dir():
        raise FileNotFoundError(f"Extracted directory not found: {required_dir}")

drive_classification_component = DRIVE_EXPERIMENT_ROOT / "classification"
drive_classification_component.mkdir(parents=True, exist_ok=True)
project_experiment_root = PROJECT_ROOT / "experiment_results" / EXPERIMENT_ID
project_experiment_root.mkdir(parents=True, exist_ok=True)
links = {
    PROJECT_ROOT / "000_dataset": local_dataset_dir,
    project_experiment_root / "segmentation": local_segmentation_component,
    project_experiment_root / "classification": drive_classification_component,
}

for link, target in links.items():
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"Symlink points to a different target: {link}")
    elif link.exists():
        raise FileExistsError(
            f"Path already exists and is not a symlink: {link}. Use a clean Colab clone."
        )
    else:
        os.symlink(target, link, target_is_directory=True)
    print(f"{link} -> {target}")

## 8. Dataset and model preflight

This cell checks every metadata path, verifies that all probability maps are present, and runs one sample through the model before long-running training starts.

In [ ]:
import importlib
import sys
import pandas as pd

project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string not in sys.path:
    sys.path.insert(0, project_root_string)
importlib.invalidate_caches()

dataset_root = local_dataset_dir / "_segmentation_dataset_v2"
metadata_path = dataset_root / "004_classification_cv_5fold_seed42.csv"
probability_root = (
    local_experiment_dir / "segmentation/unet/inference/probability_npy"
)

metadata = pd.read_csv(metadata_path)
missing_ct = [
    path for path in metadata["ct_windowed_path"]
    if not (dataset_root / str(path)).is_file()
]
missing_probability = [
    name for name in metadata["filename"]
    if not (probability_root / Path(str(name)).name).is_file()
]
missing_mask = [
    path for path in metadata["mask_path"]
    if not (dataset_root / str(path)).is_file()
]
if missing_ct or missing_probability or missing_mask:
    raise FileNotFoundError(
        f"Missing CT={len(missing_ct)}, mask={len(missing_mask)}, "
        f"probability={len(missing_probability)}"
    )

dataset_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.dataset"
)
transform_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.transforms"
)
model_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.model"
)

validation_dataset = dataset_module.ProbabilityGuidedClassificationDataset(
    root_dir=dataset_root,
    metadata_path=metadata_path,
    split="val",
    cv_fold=0,
    probability_root=probability_root,
    transform=transform_module.build_val_transform(
        224, 224, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), 42
    ),
)
sample, target = validation_dataset[0]
smoke_model = model_module.SegmentationGuidedResNet50(
    num_classes=2, weights=None
).to("cuda").eval()
with torch.no_grad():
    smoke_output = smoke_model(sample.unsqueeze(0).to("cuda"))
del smoke_model
torch.cuda.empty_cache()

print(f"Metadata rows       : {len(metadata):,}")
print(f"Validation fold 0   : {len(validation_dataset):,}")
print(f"Input sample        : {tuple(sample.shape)}")
print(f"Target              : {int(target)}")
print(f"Model output        : {tuple(smoke_output.shape)}")
print(f"Probability maps    : {len(list(probability_root.glob('*.npy'))):,}")
print(f"Ground-truth masks  : {len(list((dataset_root / 'mask').glob('*.npy'))):,}")
print("Preflight completed successfully.")

## 9. Run five-fold training

This cell updates the cloned JSON configuration with the Colab values, after which `train.py` copies the effective JSON snapshot to the Google Drive output. Each UUID has only one `classification/guided_resnet50` directory, and the script refuses to overwrite an existing directory. If an OOM occurs before training starts successfully, reduce `BATCH_SIZE` and restart the runtime to fully clear VRAM.

In [ ]:
import json

config_path = (
    PROJECT_ROOT / "003_classification/configs/"
    "segmentation_guided_cv_resnet50.json"
)
with config_path.open("r", encoding="utf-8") as file:
    training_config = json.load(file)

# Paths remain relative and identical to the local layout.
training_config["experiment"]["id"] = EXPERIMENT_ID
training_config["data"]["probability_root"] = (
    f"experiment_results/{EXPERIMENT_ID}/"
    "segmentation/unet/inference/probability_npy"
)
training_config["training"]["batch_size"] = BATCH_SIZE
training_config["training"]["num_epochs"] = NUM_EPOCHS
training_config["training"]["device"] = "cuda"
training_config["dataloader"]["num_workers"] = NUM_WORKERS
training_config["dataloader"]["persistent_workers"] = (
    NUM_WORKERS > 0
)
training_config["early_stopping"]["patience"] = (
    EARLY_STOPPING_PATIENCE
)
with config_path.open("w", encoding="utf-8") as file:
    json.dump(training_config, file, indent=4)
    file.write("\n")

importlib.invalidate_caches()
module_name = "003_classification.segmentation_guided_cv_resnet50.train"
if module_name in sys.modules:
    train_module = importlib.reload(sys.modules[module_name])
else:
    train_module = importlib.import_module(module_name)

if train_module.OUTPUT_DIR.resolve() != DRIVE_GUIDED_OUTPUT_DIR.resolve():
    raise RuntimeError(
        f"Unexpected output path: {train_module.OUTPUT_DIR}"
    )

print(f"Output run : {train_module.OUTPUT_DIR}")
print(f"Batch size : {train_module.BATCH_SIZE}")
print(f"Epoch/fold : {train_module.NUM_EPOCHS}")
print(f"Patience   : {train_module.EARLY_STOPPING_PATIENCE}")
train_module.main()

## 10. Inspect training outputs

In [ ]:
from IPython.display import display

latest_run = DRIVE_GUIDED_OUTPUT_DIR
if not latest_run.is_dir():
    raise FileNotFoundError(f"No training output found at {latest_run}")
print(f"Guided classification: {latest_run}")

summary_path = latest_run / "cv_summary.csv"
if summary_path.is_file():
    display(pd.read_csv(summary_path))
else:
    completed_folds = sorted(
        path.name for path in latest_run.glob("fold_*")
        if (path / "best_model.pth").is_file()
    )
    print(f"Training is incomplete. Folds with saved models: {completed_folds}")

## 11. Run testing, Grad-CAM, and LRP

This cell verifies that ground-truth masks are available and then evaluates `experiment_results/EXPERIMENT_ID/classification/guided_resnet50`. By default, the complete holdout set is evaluated. LRP is computationally expensive; set `MAX_TEST_SAMPLES` to a small number such as `8` for a smoke test. `TEST_NUM_WORKERS=0` improves stability on Colab/Python 3.13. Visualizations are saved as one PNG per study in `test/visualization/`, with a separate section for each nodule.

In [ ]:
import subprocess
import sys

# None selects the configured completed run. Set a Path to select another run.
TEST_RUN_DIR = None
MAX_TEST_SAMPLES = None  # example: 8 for a smoke test
TEST_BATCH_SIZE = 2
TEST_NUM_WORKERS = 0  # most stable setting for Colab/Python 3.13
TEST_DPI = 120  # suitable for potentially tall study canvases

# Study visualizations require ground-truth masks. An older archive may
# have been extracted before masks were added to the training-data list.
mask_root = dataset_root / "mask"
holdout_metadata = pd.read_csv(metadata_path)
holdout_metadata = holdout_metadata.loc[
    holdout_metadata["cv_role"].astype(str).str.lower().eq("holdout_test")
]
missing_masks = [
    dataset_root / str(path)
    for path in holdout_metadata["mask_path"]
    if not (dataset_root / str(path)).is_file()
]
if missing_masks:
    print(
        f"Extracting ground-truth masks ({len(missing_masks):,} missing)...",
        flush=True,
    )
    mask_archive_member = (
        "000_dataset/_segmentation_dataset_v2/mask"
    )
    subprocess.run(
        [
            "tar", "-xzf", str(DRIVE_DATASET_ARCHIVE),
            "-C", str(LOCAL_DATA_ROOT), mask_archive_member,
        ],
        check=True,
    )
    missing_masks = [
        dataset_root / str(path)
        for path in holdout_metadata["mask_path"]
        if not (dataset_root / str(path)).is_file()
    ]
if missing_masks:
    raise FileNotFoundError(
        f"Ground-truth masks are still missing: {len(missing_masks):,}"
    )
print(f"Ground-truth masks ready: {len(holdout_metadata):,}")

required_folds = tuple(range(5))

if TEST_RUN_DIR is None:
    test_run_dir = DRIVE_GUIDED_OUTPUT_DIR
else:
    test_run_dir = Path(TEST_RUN_DIR).expanduser().resolve()

missing_checkpoints = [
    test_run_dir / f"fold_{fold}/best_model.pth"
    for fold in required_folds
    if not (test_run_dir / f"fold_{fold}/best_model.pth").is_file()
]
if missing_checkpoints:
    raise FileNotFoundError(
        "Fold checkpoints are incomplete:\n"
        + "\n".join(str(path) for path in missing_checkpoints)
    )

command = [
    sys.executable, "-m",
    "003_classification.segmentation_guided_cv_resnet50.test",
    str(test_run_dir),
    "--batch-size", str(TEST_BATCH_SIZE),
    "--num-workers", str(TEST_NUM_WORKERS),
    "--device", "cuda",
    "--dpi", str(TEST_DPI),
]
if MAX_TEST_SAMPLES is not None:
    command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

print(f"Testing run : {test_run_dir}")
print("Running ensemble inference, Grad-CAM, and LRP...", flush=True)
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print(f"Test output         : {test_run_dir / 'test'}")
print(f"Study visualization: {test_run_dir / 'test/visualization'}")